# Support Vector Machines (SVMs) - A Classic Algorithm Worth Understanding

**Learning Objectives:**
- Understand the intuition behind **maximum margin classification**
- Learn how SVMs find the optimal decision boundary
- Understand **support vectors** and why they're special
- Master the **kernel trick** for non-linear classification
- Build strong intuitions about hyperparameter tuning (C, gamma, kernel)
- Compare SVMs with other classifiers and know when to use them

By the end of this notebook, you'll have **deep intuitions** about one of the most elegant algorithms in machine learning.


## Part 1: Why SVMs Matter

Support Vector Machines were one of the dominant algorithms in machine learning before deep learning took over. They're still incredibly relevant because:

1. **Elegant mathematical foundation** - They maximize the margin between classes
2. **Effective in high dimensions** - Work well even when features > samples
3. **Memory efficient** - Only depend on a subset of training points
4. **Versatile** - The kernel trick handles non-linear boundaries
5. **Still competitive** - Often match or beat neural networks on small/medium datasets

### Real-World Applications

- **Image classification** (before CNNs dominated)
- **Text categorization** (spam detection, sentiment analysis)
- **Bioinformatics** (cancer classification from gene expression)
- **Handwriting recognition** (digit recognition)
- **Financial forecasting** (stock prediction)


## Part 2: Setup

### Configuration

All hyperparameters in one place for easy experimentation.


In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 42,  # Random seed for reproducibility
    
    # Data
    'n_samples': 200,  # Number of samples for synthetic datasets
    'test_size': 0.2,  # Fraction of data for testing
    'noise': 0.1,  # Noise level for synthetic data
    
    # SVM Hyperparameters
    'C': 1.0,  # Regularization parameter (inverse of regularization strength)
    'kernel': 'rbf',  # Kernel type: 'linear', 'poly', 'rbf', 'sigmoid'
    'gamma': 'scale',  # Kernel coefficient for 'rbf', 'poly', 'sigmoid'
    'degree': 3,  # Degree for polynomial kernel
}


### Random Seed & Imports


In [ ]:
from aiml_notebooks import set_seed
import numpy as np

set_seed(CONFIG['seed'])

print(f"✓ Seed set to {CONFIG['seed']} for reproducibility")


## Part 3: The Classification Problem

Before diving into SVMs, let's understand the classification problem they solve.

**The question:** Given labeled data points, how do we find a decision boundary that separates the classes?

Let's create a simple 2D dataset to visualize this.


In [ ]:
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

# Generate two clusters
X, y = make_blobs(
    n_samples=CONFIG['n_samples'],
    centers=2,
    cluster_std=1.5,
    random_state=CONFIG['seed']
)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='#3498db', s=60, label='Class 0', edgecolors='black', linewidth=0.5)
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='#e74c3c', s=60, label='Class 1', edgecolors='black', linewidth=0.5)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Binary Classification Problem: Where Should the Boundary Be?', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")


### The Problem: Many Possible Boundaries

Looking at this data, we could draw **many different lines** that separate the two classes. Which one should we choose?

Let's visualize a few candidate decision boundaries.


In [ ]:
# Let's draw a few possible separating lines
plt.figure(figsize=(10, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='#3498db', s=60, label='Class 0', edgecolors='black', linewidth=0.5)
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='#e74c3c', s=60, label='Class 1', edgecolors='black', linewidth=0.5)

# Get x range
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
x_line = np.linspace(x_min, x_max, 100)

# Compute midpoint between cluster centers for reference
center_0 = X[y == 0].mean(axis=0)
center_1 = X[y == 1].mean(axis=0)
midpoint = (center_0 + center_1) / 2

# Draw several candidate lines (roughly separating)
lines = [
    (-0.3, midpoint[1] + 0.3 * midpoint[0], '#27ae60', 'Line A'),
    (-0.5, midpoint[1] + 0.5 * midpoint[0] + 1, '#8e44ad', 'Line B'),
    (-0.1, midpoint[1] + 0.1 * midpoint[0] - 1, '#f39c12', 'Line C'),
]

for slope, intercept, color, label in lines:
    y_line = slope * x_line + intercept
    plt.plot(x_line, y_line, color=color, linewidth=2, linestyle='--', label=label)

plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Multiple Lines Can Separate the Classes - Which is Best?', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.xlim(x_min, x_max)
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 🤔 Reflection Question

**Q:** All three lines separate the classes perfectly. But intuitively, which one would you trust most for new, unseen data points?

**Hint:** Think about what happens if a new point appears close to the boundary.


## Part 4: The SVM Intuition - Maximum Margin

SVMs answer this question with a beautiful idea: **choose the boundary with the largest margin**.

### What is the Margin?

The **margin** is the distance from the decision boundary to the nearest data point(s) from either class.

### Why Maximize It?

A larger margin means:
- **More confidence**: Points near the boundary are "harder" to classify
- **Better generalization**: More room for new points to fall on the correct side
- **Robustness**: Small perturbations in data don't change predictions

Think of it like drawing a thick line instead of a thin one - the thicker the "no-man's land" between classes, the safer our boundary!


### Visualizing the Margin

Let's train an SVM and visualize its decision boundary with the margin.


In [ ]:
from sklearn.svm import SVC

# Train a linear SVM
svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X, y)

print(f"✓ SVM trained successfully!")
print(f"  Number of support vectors: {len(svm_linear.support_vectors_)}")
print(f"  Support vectors per class: {svm_linear.n_support_}")


Now let's create a helper function to visualize SVM decision boundaries with margins.


In [ ]:
def plot_svm_decision_boundary(clf, X, y, title="SVM Decision Boundary", show_margin=True, ax=None):
    """Plot SVM decision boundary with margins and support vectors."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))
    
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    
    # Get predictions
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary and margins
    ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), Z.max(), 50), cmap='RdBu', alpha=0.3)
    ax.contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)  # Decision boundary
    
    if show_margin:
        ax.contour(xx, yy, Z, levels=[-1, 1], colors='black', linewidths=1, linestyles='--')  # Margins
    
    # Plot data points
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#3498db', s=60, label='Class 0', edgecolors='black', linewidth=0.5)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#e74c3c', s=60, label='Class 1', edgecolors='black', linewidth=0.5)
    
    # Highlight support vectors
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], 
               s=200, facecolors='none', edgecolors='#2ecc71', linewidths=2, 
               label=f'Support Vectors (n={len(clf.support_vectors_)})')
    
    ax.set_xlabel('Feature 1', fontsize=12)
    ax.set_ylabel('Feature 2', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    
    return ax

# Visualize
fig, ax = plt.subplots(figsize=(10, 7))
plot_svm_decision_boundary(svm_linear, X, y, "Linear SVM with Maximum Margin", ax=ax)
plt.tight_layout()
plt.show()


### Key Observations

1. **Solid black line**: The decision boundary (separating hyperplane)
2. **Dashed lines**: The margin boundaries (distance = 1 in normalized space)
3. **Green circles**: **Support vectors** - the data points that define the margin
4. **Color gradient**: How confident the SVM is about each region

Notice that the decision boundary **only depends on the support vectors**! All other points could be removed without changing the boundary.


## Part 5: What Are Support Vectors?

**Support vectors** are the data points that lie on or inside the margin boundaries. They're special because:

1. **They define the boundary**: Remove them, and the boundary changes
2. **They're the "hardest" points**: Closest to the boundary, hardest to classify
3. **Memory efficient**: Only need to store support vectors for predictions

Let's examine them more closely.


In [ ]:
# Get support vectors
support_vectors = svm_linear.support_vectors_
support_indices = svm_linear.support_
support_labels = y[support_indices]

print("Support Vector Analysis")
print("=" * 50)
print(f"\nTotal training samples: {len(X)}")
print(f"Number of support vectors: {len(support_vectors)}")
print(f"Percentage of data as SVs: {100 * len(support_vectors) / len(X):.1f}%")
print(f"\nSupport vectors per class:")
print(f"  Class 0: {sum(support_labels == 0)}")
print(f"  Class 1: {sum(support_labels == 1)}")

print(f"\nSupport vector coordinates:")
for i, (sv, label) in enumerate(zip(support_vectors, support_labels)):
    print(f"  SV {i+1}: ({sv[0]:.3f}, {sv[1]:.3f}) - Class {label}")


### 💡 Key Insight

The SVM's decision boundary is completely determined by the **support vectors**. This is why SVMs are memory-efficient - at prediction time, we only need to consider the support vectors, not the entire training set!


## Part 6: The Math Behind SVMs (Simplified)

Let's understand the math without getting lost in details.

### The Decision Boundary

A linear SVM finds a hyperplane defined by:

$$\mathbf{w} \cdot \mathbf{x} + b = 0$$

where:
- $\mathbf{w}$ = weight vector (normal to the hyperplane)
- $\mathbf{x}$ = input features
- $b$ = bias (intercept)

### The Margin

The margin is:

$$\text{margin} = \frac{2}{||\mathbf{w}||}$$

### The Optimization Problem

To **maximize the margin**, we need to **minimize** $||\mathbf{w}||$.

**Minimize:** $\frac{1}{2}||\mathbf{w}||^2$

**Subject to:** $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$ for all $i$

This constraint ensures all points are correctly classified with a margin of at least 1.


### Visualizing the Geometry

Let's extract the SVM parameters and understand the geometry.


In [ ]:
# Extract SVM parameters
w = svm_linear.coef_[0]
b = svm_linear.intercept_[0]

# Calculate margin
margin = 2 / np.linalg.norm(w)

print("SVM Geometry")
print("=" * 50)
print(f"\nWeight vector w: [{w[0]:.4f}, {w[1]:.4f}]")
print(f"Bias b: {b:.4f}")
print(f"||w||: {np.linalg.norm(w):.4f}")
print(f"Margin: {margin:.4f}")
print(f"\nDecision boundary equation:")
print(f"  {w[0]:.4f}*x1 + {w[1]:.4f}*x2 + {b:.4f} = 0")


## Part 7: Soft Margin - Handling Noisy Data

Real-world data is rarely perfectly separable. What if some points are "on the wrong side"?

The **soft margin** SVM allows some points to violate the margin, controlled by the **C parameter**.

### The C Parameter

- **High C**: Strict boundary, tries to classify all training points correctly (risk of overfitting)
- **Low C**: Relaxed boundary, allows more misclassifications (more regularization)

Let's create a noisier dataset and see how C affects the boundary.


In [ ]:
# Generate noisier, overlapping data
X_noisy, y_noisy = make_blobs(
    n_samples=CONFIG['n_samples'],
    centers=2,
    cluster_std=2.5,  # More spread
    random_state=CONFIG['seed']
)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_noisy[y_noisy == 0, 0], X_noisy[y_noisy == 0, 1], c='#3498db', s=60, label='Class 0', edgecolors='black', linewidth=0.5)
plt.scatter(X_noisy[y_noisy == 1, 0], X_noisy[y_noisy == 1, 1], c='#e74c3c', s=60, label='Class 1', edgecolors='black', linewidth=0.5)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Noisy Dataset - Classes Overlap!', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Effect of C Parameter

Let's train SVMs with different C values and compare.


In [ ]:
C_values = [0.01, 0.1, 1.0, 100]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, C in zip(axes, C_values):
    svm = SVC(kernel='linear', C=C)
    svm.fit(X_noisy, y_noisy)
    
    plot_svm_decision_boundary(
        svm, X_noisy, y_noisy,
        f'C = {C} (Support Vectors: {len(svm.support_vectors_)})',
        ax=ax
    )
    
    # Calculate training accuracy
    accuracy = svm.score(X_noisy, y_noisy)
    ax.text(0.02, 0.98, f'Train Acc: {accuracy:.1%}', transform=ax.transAxes,
            fontsize=11, verticalalignment='top', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n🔍 Key Observations:")
print("  • Low C (0.01): Wide margin, more SVs, simpler boundary")
print("  • High C (100): Narrow margin, fewer SVs, tries to fit all points")


### 💡 Key Insight: C Controls the Bias-Variance Tradeoff

| C Value | Margin | Support Vectors | Behavior |
|---------|--------|-----------------|----------|
| **Low** | Wide | Many | More regularization, simpler model |
| **High** | Narrow | Few | Less regularization, more complex model |

**Rule of thumb**: Start with C=1.0 and tune based on validation performance.


## Part 8: The Kernel Trick - Non-Linear Classification

What if data **can't** be separated by a straight line?

The **kernel trick** is one of the most elegant ideas in machine learning. It allows SVMs to find non-linear decision boundaries **without explicitly computing high-dimensional feature transformations**.

Let's see why we need this.


In [ ]:
from sklearn.datasets import make_circles

# Generate non-linearly separable data
X_circles, y_circles = make_circles(
    n_samples=CONFIG['n_samples'],
    noise=CONFIG['noise'],
    factor=0.3,
    random_state=CONFIG['seed']
)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_circles[y_circles == 0, 0], X_circles[y_circles == 0, 1], 
            c='#3498db', s=60, label='Class 0', edgecolors='black', linewidth=0.5)
plt.scatter(X_circles[y_circles == 1, 0], X_circles[y_circles == 1, 1], 
            c='#e74c3c', s=60, label='Class 1', edgecolors='black', linewidth=0.5)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Non-Linear Problem: No Line Can Separate These!', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()


### Linear SVM Fails

Let's see what happens when we try to use a linear SVM on this data.


In [ ]:
# Try a linear SVM
svm_linear_circles = SVC(kernel='linear', C=1.0)
svm_linear_circles.fit(X_circles, y_circles)

fig, ax = plt.subplots(figsize=(10, 7))
plot_svm_decision_boundary(svm_linear_circles, X_circles, y_circles, 
                           f'Linear SVM Fails! (Accuracy: {svm_linear_circles.score(X_circles, y_circles):.1%})', ax=ax)
plt.tight_layout()
plt.show()


### The Idea: Transform to Higher Dimensions

What if we could **transform** the data to a higher dimension where it becomes linearly separable?

For the circles example, consider adding a feature: $x_3 = x_1^2 + x_2^2$ (distance from origin squared).

In this 3D space, the inner circle (small $x_3$) and outer circle (large $x_3$) become separable by a plane!


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Add the new feature
X_circles_3d = np.column_stack([X_circles, X_circles[:, 0]**2 + X_circles[:, 1]**2])

# Visualize in 3D
fig = plt.figure(figsize=(14, 6))

# 2D view (original)
ax1 = fig.add_subplot(121)
ax1.scatter(X_circles[y_circles == 0, 0], X_circles[y_circles == 0, 1], 
            c='#3498db', s=60, label='Class 0', edgecolors='black', linewidth=0.5)
ax1.scatter(X_circles[y_circles == 1, 0], X_circles[y_circles == 1, 1], 
            c='#e74c3c', s=60, label='Class 1', edgecolors='black', linewidth=0.5)
ax1.set_xlabel('x₁', fontsize=12)
ax1.set_ylabel('x₂', fontsize=12)
ax1.set_title('Original 2D Space (Not Separable)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.axis('equal')

# 3D view (transformed)
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(X_circles_3d[y_circles == 0, 0], X_circles_3d[y_circles == 0, 1], X_circles_3d[y_circles == 0, 2],
            c='#3498db', s=60, label='Class 0', edgecolors='black', linewidth=0.3)
ax2.scatter(X_circles_3d[y_circles == 1, 0], X_circles_3d[y_circles == 1, 1], X_circles_3d[y_circles == 1, 2],
            c='#e74c3c', s=60, label='Class 1', edgecolors='black', linewidth=0.3)

# Draw a separating plane
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 10), np.linspace(-1.2, 1.2, 10))
zz = np.ones_like(xx) * 0.3  # Plane at z = 0.3
ax2.plot_surface(xx, yy, zz, alpha=0.3, color='green')

ax2.set_xlabel('x₁', fontsize=11)
ax2.set_ylabel('x₂', fontsize=11)
ax2.set_zlabel('x₁² + x₂²', fontsize=11)
ax2.set_title('Transformed 3D Space (Linearly Separable!)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10, loc='upper left')
ax2.view_init(elev=15, azim=45)

plt.tight_layout()
plt.show()

print("💡 In the 3D space, a simple plane (green) can separate the classes!")


### The Kernel Trick: The Magic

The **kernel trick** is brilliant because:

1. **We don't explicitly compute the transformation**: Kernels compute dot products in the transformed space directly
2. **Computationally efficient**: Even infinite-dimensional transformations are tractable!
3. **Flexible**: Different kernels = different transformations

A **kernel function** $K(x_i, x_j)$ computes the inner product $\phi(x_i) \cdot \phi(x_j)$ without ever computing $\phi(x)$ explicitly.


## Part 9: Common Kernels

Let's explore the most common kernel types.

### 1. Linear Kernel
$$K(x_i, x_j) = x_i \cdot x_j$$
- No transformation, just standard dot product
- Use when data is linearly separable

### 2. Polynomial Kernel
$$K(x_i, x_j) = (\gamma \cdot x_i \cdot x_j + r)^d$$
- Creates polynomial decision boundaries
- `degree` (d) controls complexity

### 3. RBF (Radial Basis Function) Kernel
$$K(x_i, x_j) = \exp(-\gamma ||x_i - x_j||^2)$$
- Most popular non-linear kernel
- Transforms to **infinite-dimensional** space!
- `gamma` controls the "reach" of each training point

### 4. Sigmoid Kernel
$$K(x_i, x_j) = \tanh(\gamma \cdot x_i \cdot x_j + r)$$
- Similar to neural network activation
- Less commonly used


### Comparing Different Kernels

Let's see how different kernels perform on our circles data.


In [ ]:
# Compare different kernels on the circles data
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, kernel in zip(axes, kernels):
    svm = SVC(kernel=kernel, C=1.0, gamma='scale', degree=3)
    svm.fit(X_circles, y_circles)
    
    accuracy = svm.score(X_circles, y_circles)
    
    plot_svm_decision_boundary(
        svm, X_circles, y_circles,
        f'{kernel.upper()} Kernel (Accuracy: {accuracy:.1%})',
        show_margin=(kernel == 'linear'),
        ax=ax
    )

plt.tight_layout()
plt.show()

print("\n🔍 Observations:")
print("  • Linear: Can't capture the circular boundary")
print("  • Polynomial: Can fit circular patterns with right degree")
print("  • RBF: Excellent at capturing complex boundaries")
print("  • Sigmoid: Struggles with this particular pattern")


### The RBF Kernel: A Closer Look

The **RBF (Radial Basis Function)** kernel is by far the most popular. Let's understand the `gamma` parameter.


In [ ]:
gamma_values = [0.1, 1, 10, 100]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, gamma in zip(axes, gamma_values):
    svm = SVC(kernel='rbf', C=1.0, gamma=gamma)
    svm.fit(X_circles, y_circles)
    
    accuracy = svm.score(X_circles, y_circles)
    n_sv = len(svm.support_vectors_)
    
    plot_svm_decision_boundary(
        svm, X_circles, y_circles,
        f'RBF Kernel (γ={gamma})',
        show_margin=False,
        ax=ax
    )
    
    ax.text(0.02, 0.98, f'Acc: {accuracy:.1%}\nSVs: {n_sv}', transform=ax.transAxes,
            fontsize=11, verticalalignment='top', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n🔍 Effect of gamma (γ):")
print("  • Low γ (0.1): Smooth boundary, each point has wide influence")
print("  • High γ (100): Complex boundary, each point has narrow influence")
print("\n⚠️ Very high γ can lead to overfitting (boundary hugs each point)!")


### 💡 Key Insight: Gamma Controls Flexibility

| Gamma | Influence Radius | Decision Boundary | Risk |
|-------|-----------------|-------------------|------|
| **Low** | Wide | Smooth, simple | Underfitting |
| **High** | Narrow | Complex, wiggly | Overfitting |

**Think of it as**: How much should each training point "care" about distant points?
- Low gamma: "I consider far-away points too"
- High gamma: "Only my immediate neighbors matter"


## Part 10: Hyperparameter Tuning

SVMs have three main hyperparameters to tune:

1. **C**: Regularization (soft margin)
2. **gamma**: Kernel coefficient (for RBF, poly, sigmoid)
3. **kernel**: Type of kernel function

Let's do a proper grid search with cross-validation on a real dataset.


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

# Load a real dataset
data = load_breast_cancer()
X_cancer, y_cancer = data.data, data.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=CONFIG['test_size'], random_state=CONFIG['seed']
)

# Scale features (important for SVMs!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Breast Cancer Dataset")
print(f"=" * 50)
print(f"Samples: {len(X_cancer)}")
print(f"Features: {X_cancer.shape[1]}")
print(f"Classes: {data.target_names}")
print(f"\nTrain set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")


### Grid Search for Best Hyperparameters

Let's systematically search for the best combination of C and gamma.


In [ ]:
# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'linear']
}

# Perform grid search
print("Performing Grid Search with 5-fold Cross-Validation...\n")

grid_search = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print(f"\n✓ Grid search complete!")
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")


### Visualize Grid Search Results

Let's see how different hyperparameter combinations perform.


In [ ]:
import pandas as pd

# Extract results for RBF kernel
results = pd.DataFrame(grid_search.cv_results_)
rbf_results = results[results['param_kernel'] == 'rbf']

# Create heatmap data
C_vals = param_grid['C']
gamma_vals = param_grid['gamma']
accuracy_matrix = np.zeros((len(C_vals), len(gamma_vals)))

for i, C in enumerate(C_vals):
    for j, gamma in enumerate(gamma_vals):
        mask = (rbf_results['param_C'] == C) & (rbf_results['param_gamma'] == gamma)
        if mask.any():
            accuracy_matrix[i, j] = rbf_results[mask]['mean_test_score'].values[0]

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(accuracy_matrix, cmap='YlOrRd', aspect='auto')

# Labels
ax.set_xticks(range(len(gamma_vals)))
ax.set_yticks(range(len(C_vals)))
ax.set_xticklabels(gamma_vals)
ax.set_yticklabels(C_vals)
ax.set_xlabel('gamma (γ)', fontsize=12)
ax.set_ylabel('C (regularization)', fontsize=12)
ax.set_title('RBF SVM: Cross-Validation Accuracy', fontsize=14, fontweight='bold')

# Add text annotations
for i in range(len(C_vals)):
    for j in range(len(gamma_vals)):
        text = ax.text(j, i, f'{accuracy_matrix[i, j]:.3f}',
                       ha='center', va='center', color='black', fontsize=11)

# Mark best
best_C_idx = C_vals.index(grid_search.best_params_.get('C', C_vals[0]))
best_gamma_idx = gamma_vals.index(grid_search.best_params_.get('gamma', gamma_vals[0]))
if grid_search.best_params_['kernel'] == 'rbf':
    ax.add_patch(plt.Rectangle((best_gamma_idx - 0.5, best_C_idx - 0.5), 1, 1,
                                fill=False, edgecolor='green', linewidth=3))

plt.colorbar(im, label='Accuracy')
plt.tight_layout()
plt.show()


### Evaluate on Test Set

Let's evaluate our best model on the held-out test set.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Get best model
best_svm = grid_search.best_estimator_

# Predictions
y_pred = best_svm.predict(X_test_scaled)

# Metrics
test_accuracy = best_svm.score(X_test_scaled, y_test)

print("Test Set Evaluation")
print("=" * 50)
print(f"\nBest model parameters: {grid_search.best_params_}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Number of support vectors: {len(best_svm.support_vectors_)}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))


### Confusion Matrix

Let's visualize the confusion matrix to see where our model makes mistakes.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax)

# Labels
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=data.target_names, yticklabels=data.target_names,
       ylabel='True label', xlabel='Predicted label',
       title='Confusion Matrix')

# Rotate labels
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# Add text
thresh = cm.max() / 2.
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center', fontsize=16,
                color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.show()


## Part 11: SVMs vs Other Classifiers

How do SVMs compare to other popular classifiers? Let's find out!


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import time

# Define classifiers
classifiers = {
    'SVM (RBF)': SVC(kernel='rbf', C=10, gamma=0.01),
    'SVM (Linear)': SVC(kernel='linear', C=1),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=CONFIG['seed']),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=CONFIG['seed']),
}

# Evaluate each classifier
results_list = []

print("Comparing Classifiers (5-fold CV on Breast Cancer Dataset)")
print("=" * 65)

for name, clf in classifiers.items():
    start_time = time.time()
    scores = cross_val_score(clf, X_train_scaled, y_train, cv=5, scoring='accuracy')
    elapsed = time.time() - start_time
    
    results_list.append({
        'Classifier': name,
        'Mean Accuracy': scores.mean(),
        'Std': scores.std(),
        'Time (s)': elapsed
    })
    
    print(f"{name:25s}: {scores.mean():.4f} (+/- {scores.std():.4f})  [{elapsed:.3f}s]")

# Create comparison dataframe
results_df = pd.DataFrame(results_list).sort_values('Mean Accuracy', ascending=False)
print("\n" + "=" * 65)


### Visual Comparison

Let's visualize how the classifiers compare.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Sort by accuracy
results_df_sorted = results_df.sort_values('Mean Accuracy', ascending=True)

# Color SVM differently
colors = ['#e74c3c' if 'SVM' in name else '#3498db' for name in results_df_sorted['Classifier']]

# Bar plot
bars = ax.barh(results_df_sorted['Classifier'], results_df_sorted['Mean Accuracy'], 
               color=colors, edgecolor='black', linewidth=0.5)

# Error bars
ax.errorbar(results_df_sorted['Mean Accuracy'], results_df_sorted['Classifier'],
            xerr=results_df_sorted['Std'], fmt='none', color='black', capsize=3)

ax.set_xlabel('Cross-Validation Accuracy', fontsize=12)
ax.set_title('Classifier Comparison on Breast Cancer Dataset', fontsize=14, fontweight='bold')
ax.set_xlim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis='x')

# Add accuracy labels
for bar, acc in zip(bars, results_df_sorted['Mean Accuracy']):
    ax.text(acc + 0.002, bar.get_y() + bar.get_height()/2, f'{acc:.4f}',
            va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n🔍 Key Observations:")
print(f"  • Best classifier: {results_df_sorted.iloc[-1]['Classifier']}")
print(f"  • SVMs are competitive but not always the best")
print(f"  • Performance depends heavily on the dataset")


## Part 12: When to Use SVMs

### ✅ SVMs Are Good For:

1. **High-dimensional data** (features >> samples)
   - Text classification
   - Genomics
   
2. **Small to medium datasets** (100s to 10,000s samples)
   - Computationally expensive for larger datasets
   
3. **Clear margin of separation**
   - When classes are well-separated
   
4. **When you need interpretability** (linear kernel)
   - Feature weights are directly interpretable

### ❌ SVMs Are NOT Ideal For:

1. **Very large datasets** (millions of samples)
   - Training time is O(n² to n³)
   - Use SGD-based alternatives
   
2. **Noisy data with overlapping classes**
   - Random forests often work better
   
3. **When probability estimates are needed**
   - SVM doesn't naturally produce probabilities
   - (Can use `probability=True` but it's slow)
   
4. **Images and sequences** (now)
   - Deep learning dominates these domains


## Part 13: Practical Tips

Here are hard-won tips for using SVMs effectively.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    SVM Best Practices                             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. ALWAYS SCALE YOUR FEATURES                                    ║
║     • SVMs are sensitive to feature scales                        ║
║     • Use StandardScaler or MinMaxScaler                          ║
║                                                                   ║
║  2. START WITH RBF KERNEL                                         ║
║     • Most versatile, works in most cases                         ║
║     • Try linear if features >> samples                           ║
║                                                                   ║
║  3. TUNE C AND GAMMA TOGETHER                                     ║
║     • They interact! Grid search both                             ║
║     • Start: C=[0.1, 1, 10, 100], gamma=[0.001, 0.01, 0.1, 1]     ║
║                                                                   ║
║  4. USE CROSS-VALIDATION                                          ║
║     • Never tune on test data                                     ║
║     • 5-fold CV is usually sufficient                             ║
║                                                                   ║
║  5. MONITOR NUMBER OF SUPPORT VECTORS                             ║
║     • Many SVs = overfitting risk, increase C                     ║
║     • Few SVs = might be underfitting, decrease C                 ║
║                                                                   ║
║  6. FOR LARGE DATASETS                                            ║
║     • Use LinearSVC (faster, different algorithm)                 ║
║     • Or use SGDClassifier with hinge loss                        ║
║                                                                   ║
╚══════════════════════════════════════════════════════════════════╝
""")


## Part 14: Feature Scaling Importance

Let's demonstrate why scaling matters so much for SVMs.


In [ ]:
# Train SVM without scaling
svm_no_scale = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_no_scale.fit(X_train, y_train)
acc_no_scale = svm_no_scale.score(X_test, y_test)

# Train SVM with scaling
svm_scaled = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_scaled.fit(X_train_scaled, y_train)
acc_scaled = svm_scaled.score(X_test_scaled, y_test)

print("Impact of Feature Scaling")
print("=" * 50)
print(f"\nWithout scaling: {acc_no_scale:.4f} accuracy")
print(f"With scaling:    {acc_scaled:.4f} accuracy")
print(f"\nImprovement: {(acc_scaled - acc_no_scale) * 100:.2f}%")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Without Scaling', 'With Scaling'], [acc_no_scale, acc_scaled], 
              color=['#e74c3c', '#27ae60'], edgecolor='black', linewidth=2)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('SVM Performance: Scaled vs Unscaled Features', fontsize=14, fontweight='bold')
ax.set_ylim(0.8, 1.0)
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, [acc_no_scale, acc_scaled]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{acc:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## Part 15: Multiclass Classification

SVMs are inherently binary classifiers, but they can handle multiple classes using:

1. **One-vs-One (OvO)**: Train $\frac{k(k-1)}{2}$ classifiers for k classes
2. **One-vs-Rest (OvR)**: Train k classifiers, each separating one class from the rest

Scikit-learn uses OvO by default for SVC. Let's see it in action.


In [ ]:
from sklearn.datasets import load_iris

# Load iris dataset (3 classes)
iris = load_iris()
X_iris, y_iris = iris.data[:, :2], iris.target  # Use first 2 features for visualization

# Train multiclass SVM
svm_multi = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_multi.fit(X_iris, y_iris)

# Create mesh grid for decision boundary
x_min, x_max = X_iris[:, 0].min() - 0.5, X_iris[:, 0].max() + 0.5
y_min, y_max = X_iris[:, 1].min() - 0.5, X_iris[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = svm_multi.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
ax.contour(xx, yy, Z, colors='black', linewidths=0.5)

# Plot data points
colors = ['#e74c3c', '#3498db', '#27ae60']
for i, (name, color) in enumerate(zip(iris.target_names, colors)):
    ax.scatter(X_iris[y_iris == i, 0], X_iris[y_iris == i, 1], 
               c=color, s=60, label=name, edgecolors='black', linewidth=0.5)

# Highlight support vectors
ax.scatter(svm_multi.support_vectors_[:, 0], svm_multi.support_vectors_[:, 1],
           s=150, facecolors='none', edgecolors='black', linewidths=2,
           label=f'Support Vectors (n={len(svm_multi.support_vectors_)})')

ax.set_xlabel(iris.feature_names[0], fontsize=12)
ax.set_ylabel(iris.feature_names[1], fontsize=12)
ax.set_title('Multiclass SVM on Iris Dataset', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining accuracy: {svm_multi.score(X_iris, y_iris):.4f}")
print(f"Number of support vectors per class: {svm_multi.n_support_}")


## Part 16: Key Takeaways

Congratulations! You now have deep intuitions about Support Vector Machines.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    KEY TAKEAWAYS                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. MAXIMUM MARGIN PRINCIPLE                                      ║
║     • SVMs find the boundary with largest margin between classes  ║
║     • Larger margin = better generalization                       ║
║                                                                   ║
║  2. SUPPORT VECTORS                                               ║
║     • Only points on/inside the margin define the boundary        ║
║     • Memory efficient: only need to store these                  ║
║                                                                   ║
║  3. THE KERNEL TRICK                                              ║
║     • Projects data to higher dimensions without explicit compute ║
║     • RBF kernel: most versatile, start here                      ║
║     • Linear kernel: fast, interpretable, good for many features  ║
║                                                                   ║
║  4. KEY HYPERPARAMETERS                                           ║
║     • C: tradeoff between margin size and misclassification       ║
║     • gamma: influence radius for each training point             ║
║     • Both need tuning! Use grid search with cross-validation     ║
║                                                                   ║
║  5. PRACTICAL ADVICE                                              ║
║     • ALWAYS scale features first                                 ║
║     • Best for small-medium datasets                              ║
║     • Use LinearSVC for large datasets                            ║
║     • Monitor number of support vectors                           ║
║                                                                   ║
╚══════════════════════════════════════════════════════════════════╝
""")


### 🤔 Final Reflection Questions

1. **Why** does maximizing the margin lead to better generalization?

2. **How** does the kernel trick allow infinite-dimensional transformations to be computed efficiently?

3. **When** would you choose a linear kernel over RBF?

4. **What** happens to the number of support vectors as you increase C? Why?

5. **Why** are SVMs less popular now than before deep learning? Where do they still shine?

---

**Well done!** 🎉 You now understand one of the most elegant algorithms in machine learning. SVMs may no longer dominate, but their mathematical beauty and intuitions are timeless.

---

## THE END

Check my repo for more **AI/ML** notebooks: https://github.com/tsilva/aiml-notebooks
